# Genie — DAIS 2026 Runbook

Story line: ops manager needs answers before a board call. No SQL, no analyst.

**Message:** One governed chat surface across your entire data estate — desktop and mobile.

> [The next generation of Databricks Genie](https://www.databricks.com/blog/next-generation-databricks-genie)

## Pre-flight

- 3 spaces in the workspace: `Revenue & Orders Intelligence`, `Operations Intelligence`, `Menu & Safety Intelligence`
- `<catalog>-genie-warehouse` running (warm with one sample question click)
- `/one` loads under your identity, spaces visible
- 3 Discover Domains exist (Operations, Revenue & Customers, Compliance & Safety) — Beta, manual UI setup, see `SETUP.ipynb`
- Genie iOS/Android app installed and logged in
- PDF ready to upload — a food safety report, compliance doc, or supplier notice (anything plausible for the story)

Run the next cell for fresh URLs.

In [ ]:
# Pre-flight: print fresh URLs for the demo.
# Re-run any time IDs go stale (e.g. after redeploy).

try:
    CATALOG = dbutils.widgets.get("CATALOG") or "oleksandra"
except Exception:
    CATALOG = "oleksandra"

import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")

print(f"Catalog: {CATALOG}")
print(f"Host:    {host}")
print(f"Genie app (next-gen, replaces Databricks One): {host}/one\n")

# 3 Genie Spaces (resolved from uc_state)
TITLES = {
    f"Revenue & Orders Intelligence ({CATALOG})",
    f"Operations Intelligence ({CATALOG})",
    f"Menu & Safety Intelligence ({CATALOG})",
}
print("Genie Spaces")
print("-" * 80)
seen = set()
df = spark.sql(f"""
    SELECT resource_data FROM {CATALOG}._internal_state.resources
    WHERE resource_type = 'genie_spaces' ORDER BY created_at DESC
""")
for row in df.collect():
    info = json.loads(row.resource_data)
    title = info.get("title")
    if title in seen or title not in TITLES:
        continue
    seen.add(title)
    print(f"  {title}\n    {host}/genie/rooms/{info['space_id']}")

# Shared SQL warehouse
print("\nWarehouse")
print("-" * 80)
wh_name = f"{CATALOG}-genie-warehouse"
for wh in w.warehouses.list():
    if wh.name == wh_name:
        print(f"  {wh.name}  [{wh.state}]  {host}/sql/warehouses/{wh.id}")
        break

## The demo

Open **Revenue & Orders Intelligence**. New conversation.

```
What is total revenue by location this month?
```
Show **View SQL** — auditable, gold table, runs as the asker's UC identity.

```
Which location is trending down the most compared to last month?
```
Context carries — Genie knows "location" from the previous answer. This is conversational BI, not one-shot SQL.

More questions for this space:
```
Show me the top 5 brands by order volume today.
```
```
What's the average order value across all locations this week?
```

---

Switch to **Operations Intelligence**. New conversation.

```
Which locations have a food safety score below 80?
```
```
Which location has the highest order cancellation rate this month?
```
```
What are the most common violation categories across all locations?
```

---

Switch to **Menu & Safety Intelligence**. New conversation.

```
What are our highest-protein items under $15?
```
```
Which brands have the most allergen-free options?
```
```
Show me all critical violations from the last 60 days.
```

---

Switch to **`/one`**. New conversation.

Home screen: every space, dashboard, and app the user is entitled to — across workspaces, the whole account.

```
Which Casper's location has the worst combination of declining revenue and low food safety score this month?
```
Expand the response — point at the spaces Genie consulted. Revenue and food safety live in separate spaces; the user never had to know that.

```
Give me a full health check on that location — revenue trend, cancellation rate, and latest food safety violations.
```

```
Which brand should we consider dropping based on low order volume and high complaint rate?
```

Follow-up (same conversation):
```
What are the most recent critical violations at that location?
```

---

### Discover Domains — business-aligned grouping (Beta)

Catalog → **Discover** → **Domains**. Three domains pre-curated for Casper's:

- **Operations** — orders, deliveries, locations, inspections; Ops dashboards; Operations Intelligence Genie; Ops Dashboard app
- **Revenue & Customers** — revenue/brands/customers tables; Revenue & Orders Genie; AI Agent Performance dashboard
- **Compliance & Safety** — food-safety reports + ai_parsed tables; Menu & Safety Genie; Legal / Regulatory / Audit / Inspection KAs

Click into **Operations** — every asset an ops manager needs is on one page: tables, dashboards, a Genie space, and the app, regardless of which catalog or schema they physically live in. The same `food_safety` tables appear in **Operations** *and* **Compliance & Safety** — assets aren't trapped in one folder.

**The message:** Catalogs/schemas model the data; Domains model the business. Discovery, ownership, certification, and request-for-access live where the work happens, not where the bytes happen. Beta today, no public API yet, so this was set up manually once (see `SETUP.ipynb`).

---
TO BE UPDATE - 27/05 - works only on Logfood

Pull out your phone. Open the Genie app. The same conversation is there — same context, same answers, continue it:

```
What should I prioritize fixing this week?
```

---

### Benchmarks — validate the space

Some spaces contain benchmarks, run those. If the space doesn't contain benchmarks, use Genie Code to generate benchmarks. Use the following prompt:

```
Generate 5 benchmark questions for evaluating this space. Cover:
1. A simple aggregation (e.g. total revenue, count of orders)
2. A top-N filter (e.g. top locations by a metric)
3. A time comparison (e.g. this week vs last week)
4. A JOIN across two tables returning human-readable names
5. A derived metric (e.g. cancellation rate, average order value, pass rate)

Rules:
- Questions must be natural business language, no SQL jargon
- Always JOIN dimension tables (locations, brands) to return names not IDs

```

Paste each pair into the space via Benchmarks tab → Add benchmark.

---

### What else Genie can do

- **Scheduled digests** — daily/weekly summaries pushed to email or Slack
- **MCP server** — external AI agents can query your Genie spaces as a tool
- **Document connectors** — attach Google Drive, SharePoint
- **Embed in apps** — Genie spaces are an API; route questions to them from any Databricks App or external surface
- **Account-level** — spans workspaces; one surface for the whole data estate
- **Governance throughout** — every query runs as the asker's UC identity, RLS/CLS apply

---

*"This is what governed self-service BI looks like in 2026 — chat from your desk or your phone, against all your data, with UC governance throughout."*

## Between runs

- New chat in each space and in `/one` before the next slot.
- Warehouse auto-stops — warm with one sample question click.
- Pre-flight cell prints nothing: wrong `CATALOG` or `Genie_Spaces` task hasn't run. Re-run from the `caspers` job (`-t all`).

Source of truth for the spaces: `stages/genie_spaces.ipynb`.